# Tidal analysis — The Battery, NY

Fit a TSGAM harmonic model to NOAA 8518750 water level data and evaluate on a held-out window.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path("../src")))
sys.path.insert(0, ".")

from tsgam_estimator import (
    TsgamEstimator,
    TsgamEstimatorConfig,
    TsgamMultiPeriodicConfig,
    TsgamSolverConfig,
)
from example_tidal import (
    load_station,
    STATION_CATALOG,
    TIDAL_CONSTITUENT_PERIODS_HOURS as PERIODS,
)
from tidal_analysis_helpers import compute_periodogram, infer_samples_per_hour
from tidal_model_shared import tidal_metrics

TRAIN_END = "2024-01-01"
TEST_END = "2024-04-01"

sns.set_theme(style="whitegrid", rc={
    "figure.figsize": (14, 4),
    "figure.constrained_layout.use": True,
    "grid.alpha": 0.3,
})

In [ ]:
CONSTITUENTS = {
    "M2": (PERIODS["M2"], 4),
    "S2": (PERIODS["S2"], 1),
    "N2": (PERIODS["N2"], 1),
    "K1": (PERIODS["K1"], 2),
    "O1": (PERIODS["O1"], 1),
    "Mf": (PERIODS["Mf"], 1),
    "Mm": (PERIODS["Mm"], 1),
    "annual": (8766.0, 2),
}

_desc = {
    "M2": "Principal lunar semidiurnal",
    "S2": "Principal solar semidiurnal",
    "N2": "Larger lunar elliptic semidiurnal",
    "K1": "Lunisolar diurnal",
    "O1": "Principal lunar diurnal",
    "Mf": "Lunar fortnightly",
    "Mm": "Lunar monthly",
    "annual": "Annual + semiannual",
}
pd.DataFrame(
    [(k, f"{p:.2f} h", n, _desc[k]) for k, (p, n) in CONSTITUENTS.items()],
    columns=["Constituent", "Period", "Harmonics", ""],
).set_index("Constituent")

In [ ]:
df = load_station("Battery")
df = df[:TEST_END]
sph = infer_samples_per_hour(df.index)
df.head()

---
## Look at the data

In [ ]:
met_cols = [c for c in ["pressure", "water_temp"] if c in df.columns]
n_rows = 1 + len(met_cols)
fig, axes = plt.subplots(n_rows, 1, figsize=(14, 3 * n_rows), sharex=True)
if n_rows == 1:
    axes = [axes]

axes[0].plot(df.index, df["water_level"], lw=0.3)
axes[0].axvline(pd.Timestamp(TRAIN_END).to_pydatetime(), color="k", ls="--", lw=0.8, label="train / test")
axes[0].set_ylabel("Water level (m, MLLW)")
axes[0].legend(loc="upper right")

colors = {"pressure": "tab:orange", "water_temp": "tab:red"}
labels = {"pressure": "Pressure (hPa)", "water_temp": "Water temp (°C)"}
for ax, col in zip(axes[1:], met_cols):
    ax.plot(df.index, df[col], lw=0.3, color=colors.get(col, "tab:blue"))
    ax.set_ylabel(labels.get(col, col))

plt.show()

Compute periodogram with overlayed tidal components.

In [ ]:
spec = compute_periodogram(df.index, df["water_level"], min_period_hours=4, max_period_hours=800)

fig, ax = plt.subplots()
ax.semilogy(spec["period_hours"], spec["power"], lw=0.5)
for name, (p, _) in CONSTITUENTS.items():
    if p < spec["period_hours"].max():
        ax.axvline(p, color="tab:red", ls="--", lw=0.7, alpha=0.6)
        ax.text(p, 0.95, name, transform=ax.get_xaxis_transform(),
                fontsize=8, color="tab:red", ha="center", va="top")
ax.set_xlabel("Period (hours)")
ax.set_ylabel("Power")
plt.show()

---
## Interactive explorer

Pick constituents and date ranges, hit **Run** to refit.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

harmonic_sliders = {
    name: widgets.IntSlider(value=n, min=0, max=6, description=name, style={"description_width": "60px"})
    for name, (_, n) in CONSTITUENTS.items()
}
train_start_w = widgets.DatePicker(value=df.index[0].date(), description="Train start")
train_end_w = widgets.DatePicker(value=pd.Timestamp(TRAIN_END).date(), description="Train end")
test_end_w = widgets.DatePicker(value=pd.Timestamp(TEST_END).date(), description="Test end")
run_btn = widgets.Button(description="Run", button_style="primary")
output = widgets.Output()

def _run(_):
    with output:
        clear_output(wait=True)
        picked = {k: (CONSTITUENTS[k][0], s.value)
                  for k, s in harmonic_sliders.items() if s.value > 0}
        if not picked:
            print("Set at least one constituent to harmonics > 0.")
            return

        df_w = df[train_start_w.value : test_end_w.value]
        sph_w = infer_samples_per_hour(df_w.index)
        split_w = pd.Timestamp(train_end_w.value)
        tr, te = df_w[df_w.index < split_w], df_w[df_w.index >= split_w]

        ok_tr = tr["water_level"].notna()
        X_tr = pd.DataFrame(index=tr.index[ok_tr])
        y_tr = tr.loc[ok_tr, "water_level"].values

        cfg = TsgamEstimatorConfig(
            multi_periodic_config=TsgamMultiPeriodicConfig(
                periods=[p * sph_w for p, _ in picked.values()],
                num_harmonics=[n for _, n in picked.values()],
                reg_weight=1e-4,
            ),
            exog_config=None,
            solver_config=TsgamSolverConfig(solver="SCS", verbose=False),
        )
        mdl = TsgamEstimator(cfg)
        mdl.fit(X_tr, y_tr)

        y_hat_te = mdl.predict(pd.DataFrame(index=te.index))
        ok_te = te["water_level"].notna()
        y_hat_tr = mdl.predict(pd.DataFrame(index=tr.index))
        m_tr = tidal_metrics(y_tr, y_hat_tr[ok_tr])
        m_te = tidal_metrics(te.loc[ok_te, "water_level"].values, y_hat_te[ok_te])

        print(f"Constituents: {', '.join(picked)}  |  "
              f"Train {len(y_tr):,} / Test {ok_te.sum():,} samples")
        display(pd.DataFrame({"train": m_tr, "test": m_te}).T)

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
        ax1.plot(te.index, te["water_level"], lw=0.4, label="observed", alpha=0.8)
        ax1.plot(te.index, y_hat_te, lw=0.4, label="predicted", alpha=0.8)
        ax1.set_ylabel("Water level (m)")
        ax1.legend()
        ax1.set_title(f"RMSE {m_te['rmse']:.4f} m  |  R\u00b2 {m_te['r2']:.4f}")

        resid = te["water_level"].values - y_hat_te
        ax2.plot(te.index, resid, lw=0.3, color="tab:green")
        ax2.axhline(0, color="k", lw=0.5)
        ax2.set_ylabel("Residual (m)")
        plt.show()

        fig, ax = plt.subplots(figsize=(5, 5))
        y_obs = te.loc[ok_te, "water_level"].values
        y_pred = y_hat_te[ok_te]
        ax.scatter(y_pred, y_obs, s=0.5, alpha=0.2)
        lo, hi = min(y_pred.min(), y_obs.min()), max(y_pred.max(), y_obs.max())
        ax.plot([lo, hi], [lo, hi], "k--", lw=0.5)
        ax.set_xlabel("Predicted (m)")
        ax.set_ylabel("Observed (m)")
        ax.set_aspect("equal")
        plt.show()

run_btn.on_click(_run)
display(
    widgets.HBox([
        widgets.VBox(list(harmonic_sliders.values()), layout=widgets.Layout(margin="0 20px 0 0")),
        widgets.VBox([train_start_w, train_end_w, test_end_w, run_btn]),
    ]),
    output,
)
